# P2: rule vs model, side by side (Lab 4)

Question: *which best sellers, at which stores, will run out before the next delivery, so need action today?*

Every step calls the same functions as `python -m vastra_p2 contest`, so the notebook and the job can never disagree. Run the data platform first (`python -m vastra_data pipeline`).

In [ ]:
import os, sys
sys.path.insert(0, os.path.abspath('..'))
from vastra_p2 import config, features as F, model as M, explain
import pandas as pd
pd.set_option('display.width', 160)
inputs = F.load_inputs(actor='notebook')   # every read is logged in gov.access_log for purpose 'replenishment'
df = F.contest_rows(F.build(inputs))
print(f"{len(df):,} SKU-store-days; stock-out within 7 days: {df[F.LABEL].mean():.1%}")
df.groupby('store_tier')[F.LABEL].mean().round(3)

## Three rules against leakage
1. Only what was known at 05:59: every feature for day *d* uses data up to the end of *d-1*.
2. Split by time: train on older months, test on the last 8 weeks (Dussehra and Diwali inside), with a 7-day gap.
3. Averages from past days only (`rolling(...).shift(1)`).

In [ ]:
train, test, cutoff = M.time_split(df)
print('train up to', train.business_date.max(), '| test from', cutoff, 'to', test.business_date.max())

## The two candidates
The rule flags a SKU-store if days of cover < supplier lead time + 2 safety days. The model is a depth-4 decision tree with balanced classes.

In [ ]:
model = M.train(train)
scored = M.score(model, test)
metrics = M.evaluate(scored)
pd.DataFrame(metrics['results'])[['group','stockouts','rule_recall','model_recall','rule_precision','model_precision']]

## Fairness and the acceptance gate
Look for any tier where the model is worse than the rule, or far behind the other tiers.

In [ ]:
from vastra_p2 import card
gate = M.gate(metrics, signed=card.signoff() is not None)
for c in gate['checks']: print('PASS' if c['passed'] else 'FAIL', c['check'], '->', c['evidence'])
print('\nDECISION:', gate['decision'])

## Remedy to try: give tier-3 rows more weight
Section 7.1: if a tier does worse, check its adjusted demand, weight its rows, or use a separate threshold. Here we try the weight and see what happens to every tier.

In [ ]:
w = M.evaluate(M.score(M.train(train, weights={'tier-3': 2.0}), test))
pd.DataFrame(w['results'])[['group','rule_recall','model_recall']]

## What the model relies on (7.2)
If a store or SKU id were near the top, we would suspect leakage or memorising.

In [ ]:
imp = M.feature_importance(model)
print(M.importance_warning(imp) or 'no id-like feature near the top')
imp

## Three reasons for one row
Reasons describe how the model behaves, not proof of cause.

In [ ]:
top = scored.sort_values('model_score', ascending=False).head(3)
for (_, r), rs in zip(top.iterrows(), explain.reasons(model, top)):
    print(r.business_date, r.sku_id, r.location_id, f'risk {r.risk:.2f}:', ' · '.join(rs))